# EuroSAT Land Classification — RGB vs Multispectral Analysis

This notebook compares three deep learning approaches for land use classification using the EuroSAT satellite imagery dataset.

**Models compared:**
- Model 1: Simple CNN trained on RGB images (3 bands)
- Model 2: ResNet50 Feature Extraction on RGB images (3 bands)
- Model 3: Simple CNN trained on Multispectral images (13 bands)

**Dataset:** EuroSAT — 27,000 Sentinel-2 satellite image patches across 10 land use classes

**Research Questions:**
1. Does ImageNet transfer learning help on satellite imagery?
2. Do extra spectral bands beyond RGB improve classification accuracy?

## 1. Setup & Data Loading

In [ ]:
# Install required libraries
!pip install kagglehub rasterio --quiet

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import rasterio
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import confusion_matrix
from google.colab import drive

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Mount Google Drive for saving models and results
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/eurosat', exist_ok=True)

# Download EuroSAT dataset from Kaggle
import kagglehub
path = kagglehub.dataset_download("apollo2506/eurosat-dataset")
print(f"Dataset path: {path}")

# Define paths
eurosat_path  = os.path.join(path, 'EuroSAT')
allbands_path = os.path.join(path, 'EuroSATallBands')

In [ ]:
# Load pre-split CSV files
train_df = pd.read_csv(os.path.join(eurosat_path, 'train.csv'))
val_df   = pd.read_csv(os.path.join(eurosat_path, 'validation.csv'))
test_df  = pd.read_csv(os.path.join(eurosat_path, 'test.csv'))

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"\nClasses: {sorted(train_df['ClassName'].unique())}")
print(f"\nSample row:\n{train_df.head(2)}")

## 2. Data Visualization

Visualizing one sample image from each of the 10 land use classes.

In [ ]:
import cv2

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()
classes = sorted(train_df['ClassName'].unique())

for i, cls in enumerate(classes):
    row = train_df[train_df['ClassName'] == cls].iloc[0]
    img_path = os.path.join(eurosat_path, row['Filename'])
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[i].imshow(img)
    axes[i].set_title(cls, fontsize=9)
    axes[i].axis('off')

plt.suptitle('EuroSAT — Sample Images per Class (RGB)', fontsize=13)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/eurosat/sample_images.png', dpi=150)
plt.show()

## 3. Model 1 — Simple CNN on RGB Images

**Architecture:** 3 Conv blocks (32→64→128 filters) + BatchNorm + Dropout  
**Input:** 64×64×3 RGB images  
**Purpose:** Establish baseline accuracy

In [ ]:
# Data generators for RGB (64x64)
datagen = ImageDataGenerator(rescale=1./255)

train_generator = datagen.flow_from_dataframe(
    dataframe=train_df, directory=eurosat_path,
    x_col='Filename', y_col='ClassName',
    target_size=(64, 64), batch_size=32,
    class_mode='sparse', shuffle=True
)
val_generator = datagen.flow_from_dataframe(
    dataframe=val_df, directory=eurosat_path,
    x_col='Filename', y_col='ClassName',
    target_size=(64, 64), batch_size=32,
    class_mode='sparse', shuffle=False
)
test_generator = datagen.flow_from_dataframe(
    dataframe=test_df, directory=eurosat_path,
    x_col='Filename', y_col='ClassName',
    target_size=(64, 64), batch_size=32,
    class_mode='sparse', shuffle=False
)

In [ ]:
def build_cnn(input_shape=(64, 64, 3)):
    """Build a simple 3-block CNN for image classification."""
    model = models.Sequential([
        layers.Input(shape=input_shape),

        # Conv Block 1
        layers.Conv2D(32, (3,3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2,2),
        layers.Dropout(0.25),

        # Conv Block 2
        layers.Conv2D(64, (3,3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2,2),
        layers.Dropout(0.25),

        # Conv Block 3
        layers.Conv2D(128, (3,3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2,2),
        layers.Dropout(0.25),

        # Classification Head
        layers.Flatten(),
        layers.Dense(256),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ])
    return model

model = build_cnn(input_shape=(64, 64, 3))
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
# Train Simple CNN on RGB
callbacks = [
    ModelCheckpoint(
        '/content/drive/MyDrive/eurosat/best_model.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_accuracy', factor=0.5,
        patience=3, verbose=1
    )
]

history = model.fit(
    train_generator,
    epochs=20,
    validation_data=val_generator,
    callbacks=callbacks
)

# Save training history
with open('/content/drive/MyDrive/eurosat/history_simplecnn.json', 'w') as f:
    json.dump({k: [float(v) for v in vals] for k, vals in history.history.items()}, f)

# Evaluate on test set
test_loss, test_acc = model.evaluate(test_generator)
print(f"\nSimple CNN RGB — Test Accuracy: {test_acc*100:.2f}%")
print(f"Simple CNN RGB — Test Loss: {test_loss:.4f}")

In [ ]:
# Plot training curves for Simple CNN
history_data = history.history
epochs = range(1, len(history_data['accuracy']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, history_data['accuracy'], 'b-', label='Train Accuracy')
axes[0].plot(epochs, history_data['val_accuracy'], 'r-', label='Val Accuracy')
axes[0].set_title('Simple CNN RGB — Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(epochs, history_data['loss'], 'b-', label='Train Loss')
axes[1].plot(epochs, history_data['val_loss'], 'r-', label='Val Loss')
axes[1].set_title('Simple CNN RGB — Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.suptitle('Simple CNN RGB Training History', fontsize=14)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/eurosat/simplecnn_training_curves.png', dpi=150)
plt.show()

In [ ]:
# Confusion Matrix — Simple CNN RGB
class_names = ['AnnualCrop', 'Forest', 'HerbaceousVeg',
                'Highway', 'Industrial', 'Pasture',
                'PermanentCrop', 'Residential', 'River', 'SeaLake']

y_pred, y_true = [], []
for images, labels in test_generator:
    predictions = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(predictions, axis=1))
    y_true.extend(labels.astype(int))
    if len(y_true) >= len(test_df):
        break

cm = confusion_matrix(y_true[:len(test_df)], y_pred[:len(test_df)])
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix — Simple CNN RGB')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/eurosat/confusion_matrix_rgb.png', dpi=150)
plt.show()

## 4. Model 2 — ResNet50 Feature Extraction on RGB Images

**Architecture:** ResNet50 (pretrained on ImageNet, frozen) + custom classification head  
**Input:** 224×224×3 RGB images (ResNet50 requires minimum 224×224)  
**Purpose:** Test if ImageNet transfer learning helps on satellite imagery

In [ ]:
# Data generators for ResNet50 (224x224)
datagen_224 = ImageDataGenerator(rescale=1./255)

train_generator_224 = datagen_224.flow_from_dataframe(
    dataframe=train_df, directory=eurosat_path,
    x_col='Filename', y_col='ClassName',
    target_size=(224, 224), batch_size=32,
    class_mode='sparse', shuffle=True
)
val_generator_224 = datagen_224.flow_from_dataframe(
    dataframe=val_df, directory=eurosat_path,
    x_col='Filename', y_col='ClassName',
    target_size=(224, 224), batch_size=32,
    class_mode='sparse', shuffle=False
)
test_generator_224 = datagen_224.flow_from_dataframe(
    dataframe=test_df, directory=eurosat_path,
    x_col='Filename', y_col='ClassName',
    target_size=(224, 224), batch_size=32,
    class_mode='sparse', shuffle=False
)

In [ ]:
# Build ResNet50 Feature Extraction Model
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False  # Freeze all ResNet50 layers

model_fe = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

model_fe.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f"Trainable params: {sum([tf.size(w).numpy() for w in model_fe.trainable_weights]):,}")
print(f"Non-trainable params: {sum([tf.size(w).numpy() for w in model_fe.non_trainable_weights]):,}")

In [ ]:
# Train ResNet50 Feature Extraction
callbacks_fe = [
    ModelCheckpoint(
        '/content/drive/MyDrive/eurosat/best_model_fe.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_accuracy', factor=0.5,
        patience=3, verbose=1
    )
]

history_fe = model_fe.fit(
    train_generator_224,
    epochs=20,
    validation_data=val_generator_224,
    callbacks=callbacks_fe
)

# Evaluate on test set
test_loss_fe, test_acc_fe = model_fe.evaluate(test_generator_224)
print(f"\nResNet50 Feature Extraction — Test Accuracy: {test_acc_fe*100:.2f}%")
print(f"ResNet50 Feature Extraction — Test Loss: {test_loss_fe:.4f}")

## 5. Model 3 — Simple CNN on 13-band Multispectral Images

**Architecture:** Same 3-block CNN as Model 1, input changed to 13 channels  
**Input:** 64×64×13 multispectral TIF images  
**Purpose:** Test if extra spectral bands beyond RGB improve accuracy  
**Key difference:** Uses custom `tf.data` pipeline with `rasterio` to load TIF files

In [ ]:
# Convert RGB CSV filenames (.jpg) to multispectral (.tif)
train_df_13 = train_df.copy()
val_df_13   = val_df.copy()
test_df_13  = test_df.copy()

train_df_13['Filename'] = train_df_13['Filename'].str.replace('.jpg', '.tif')
val_df_13['Filename']   = val_df_13['Filename'].str.replace('.jpg', '.tif')
test_df_13['Filename']  = test_df_13['Filename'].str.replace('.jpg', '.tif')

print(f"Sample filename: {train_df_13['Filename'].iloc[0]}")

In [ ]:
# Inspect one TIF file to understand format
sample_tif = os.path.join(allbands_path, train_df_13['Filename'].iloc[0])
with rasterio.open(sample_tif) as src:
    data = src.read()
    print(f"Bands: {src.count}")
    print(f"Shape: {data.shape}  (bands, height, width)")
    print(f"dtype: {src.dtypes[0]}")
    print(f"Value range: {data.min()} — {data.max()}")
    print(f"\nNote: uint16 max = 65535 → normalize by dividing by 65535")

In [ ]:
# Custom tf.data pipeline for 13-band TIF files

def load_tif(filename, label):
    """Load a single TIF file and preprocess it."""
    filename = filename.numpy().decode('utf-8')
    tif_path = os.path.join(allbands_path, filename)
    with rasterio.open(tif_path) as src:
        data = src.read()                        # (13, 64, 64)
    data = data.transpose(1, 2, 0)               # (64, 64, 13)
    data = data.astype(np.float32) / 65535.0     # normalize to [0, 1]
    return data, label

def tf_load_tif(filename, label):
    """TensorFlow-compatible wrapper for load_tif."""
    image, label = tf.py_function(
        func=load_tif,
        inp=[filename, label],
        Tout=[tf.float32, tf.int64]
    )
    image.set_shape([64, 64, 13])
    label.set_shape([])
    return image, label

def create_dataset_13band(df, shuffle=False, batch_size=32):
    """Build a tf.data pipeline for 13-band TIF images."""
    filenames = df['Filename'].values
    labels    = df['Label'].values
    dataset = tf.data.Dataset.from_tensor_slices((filenames, labels))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=1000)
    dataset = dataset.map(tf_load_tif, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

train_dataset_13 = create_dataset_13band(train_df_13, shuffle=True)
val_dataset_13   = create_dataset_13band(val_df_13)
test_dataset_13  = create_dataset_13band(test_df_13)

print("13-band datasets ready")

In [ ]:
# Build 13-band CNN — identical to Simple CNN except input shape
model_13 = build_cnn(input_shape=(64, 64, 13))
model_13.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model_13.summary()

In [ ]:
# Train 13-band CNN
callbacks_13 = [
    ModelCheckpoint(
        '/content/drive/MyDrive/eurosat/best_model_13band.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_accuracy', factor=0.5,
        patience=3, verbose=1
    )
]

history_13 = model_13.fit(
    train_dataset_13,
    epochs=20,
    validation_data=val_dataset_13,
    callbacks=callbacks_13
)

# Save training history
with open('/content/drive/MyDrive/eurosat/history_13band.json', 'w') as f:
    json.dump({k: [float(v) for v in vals] for k, vals in history_13.history.items()}, f)

# Evaluate on test set
test_loss_13, test_acc_13 = model_13.evaluate(test_dataset_13)
print(f"\n13-band CNN — Test Accuracy: {test_acc_13*100:.2f}%")
print(f"13-band CNN — Test Loss: {test_loss_13:.4f}")

In [ ]:
# Plot training curves for 13-band CNN
history_data_13 = history_13.history
epochs_13 = range(1, len(history_data_13['accuracy']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_13, history_data_13['accuracy'], 'b-', label='Train Accuracy')
axes[0].plot(epochs_13, history_data_13['val_accuracy'], 'r-', label='Val Accuracy')
axes[0].set_title('13-band CNN — Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(epochs_13, history_data_13['loss'], 'b-', label='Train Loss')
axes[1].plot(epochs_13, history_data_13['val_loss'], 'r-', label='Val Loss')
axes[1].set_title('13-band CNN — Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.suptitle('13-band CNN Training History', fontsize=14)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/eurosat/13band_training_curves.png', dpi=150)
plt.show()

In [ ]:
# Confusion Matrix — 13-band CNN
y_pred_13, y_true_13 = [], []
for images, labels in test_dataset_13:
    predictions = model_13.predict(images, verbose=0)
    y_pred_13.extend(np.argmax(predictions, axis=1))
    y_true_13.extend(labels.numpy())

cm_13 = confusion_matrix(y_true_13, y_pred_13)
plt.figure(figsize=(12, 10))
sns.heatmap(cm_13, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix — 13-band CNN')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/eurosat/confusion_matrix_13band.png', dpi=150)
plt.show()

## 6. Results Comparison

Final comparison of all three models.

In [ ]:
# Final Results Summary
print("=" * 55)
print("          FINAL RESULTS SUMMARY")
print("=" * 55)
print(f"{'Model':<35} {'Test Accuracy':>15}")
print("-" * 55)
print(f"{'Simple CNN (RGB 64x64x3)':<35} {test_acc*100:>14.2f}%")
print(f"{'ResNet50 Feature Extraction (RGB)':<35} {test_acc_fe*100:>14.2f}%")
print(f"{'Simple CNN (13-band 64x64x13)':<35} {test_acc_13*100:>14.2f}%")
print("=" * 55)
print(f"\nMultispectral improvement over RGB: +{(test_acc_13 - test_acc)*100:.1f}%")

In [ ]:
# Bar chart comparison
models_list = ['Simple CNN\n(RGB)', 'ResNet50 FE\n(RGB)', 'Simple CNN\n(13-band)']
accuracies  = [test_acc*100, test_acc_fe*100, test_acc_13*100]
colors      = ['#4C72B0', '#DD8452', '#55A868']

plt.figure(figsize=(8, 6))
bars = plt.bar(models_list, accuracies, color=colors, width=0.5, edgecolor='black')

for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{acc:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.ylim(70, 100)
plt.ylabel('Test Accuracy (%)', fontsize=12)
plt.title('Model Comparison — EuroSAT Land Classification', fontsize=13)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/eurosat/model_comparison.png', dpi=150)
plt.show()

## 7. Key Findings

**Finding 1 — ImageNet transfer learning hurts on satellite data:**  
ResNet50 with frozen ImageNet weights (80%) performed worse than a simple CNN trained from scratch (85%). The domain gap between natural photos and satellite imagery is too large for direct feature transfer.

**Finding 2 — Multispectral bands dramatically improve accuracy:**  
Adding 10 extra spectral bands (NIR, Red Edge, SWIR) beyond RGB improved accuracy by ~10%, from 85% to 95.5%.

**Finding 3 — Spectrally similar classes benefit most:**  
Classes that look identical in RGB (Pasture vs Forest, SeaLake vs Forest) became separable with multispectral data because they reflect differently in non-visible wavelengths.